# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print metadata name and description
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Authors: {[author for author in getattr(metadata, 'author', [])]}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets by their `@id`
print("Available record sets (by '@id'):")
record_set_objs = list(dataset.metadata.record_sets)
record_set_ids = []
for recset in record_set_objs:
    print(f"- {recset.id} (name: {getattr(recset, 'name', '')})")
    record_set_ids.append(recset.id)

# For each record set, list its fields (by @id) and columns
for recset in record_set_objs:
    print(f"\nRecord Set: {recset.id} (name: {getattr(recset, 'name', '')})")
    if hasattr(recset, 'fields'):
        print("  Fields:")
        for f in recset.fields:
            field_id = getattr(f, 'id', None)
            field_name = getattr(f, 'name', None)
            field_type = getattr(f, 'data_type', None)
            print(f"    - {field_id} (name: {field_name}, type: {field_type})")
            # If field has columns
            columns = getattr(f, 'columns', [])
            if columns:
                print("      Columns:")
                for col in columns:
                    col_id = getattr(col, 'id', None)
                    col_name = getattr(col, 'name', None)
                    col_type = getattr(col, 'data_type', None)
                    print(f"        - {col_id} (name: {col_name}, type: {col_type})")
    else:
        print("  (No fields listed.)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, select one or more record sets by their @id
if not record_set_ids:
    raise Exception("No record sets found in the dataset.")
selected_record_sets = record_set_ids  # Use all found

dataframes = {}
for recset_id in selected_record_sets:
    try:
        records = list(dataset.records(record_set=recset_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[recset_id] = df
            print(f"\nColumns for record set {recset_id}:\n{df.columns.tolist()}")
            print(f"First 5 records for {recset_id}:")
            display(df.head())
        else:
            print(f"No records found for record set {recset_id}")
    except Exception as e:
        print(f"Error loading records for record set {recset_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")  # just for notebook cleanliness

# Choose one record set for analysis
record_set_id = selected_record_sets[0]  # Pick the first for demonstration
df = dataframes[record_set_id]
print(f"Analyzing data for record set: {record_set_id}\n")

# Heuristically pick a numeric field (search for float columns)
numeric_candidate = None
for col in df.columns:
    # Try to convert to numeric
    try:
        ser = pd.to_numeric(df[col], errors='coerce')
        if ser.notnull().mean() > 0.7 and ser.nunique() > 10:
            numeric_candidate = col
            break
    except:
        continue

# If no such column, prompt the column list for manual pick
if numeric_candidate is None:
    numeric_candidate = df.select_dtypes(include=[np.number]).columns
    if len(numeric_candidate)>0:
        numeric_candidate = numeric_candidate[0]
    else:
        print(df.dtypes)
        raise Exception('No numeric field detected, please select one matching the dataset')

print(f"Using numeric field '{numeric_candidate}' for filtering and normalization.\n")
# Ensure it's float for easy analysis
df[numeric_candidate] = pd.to_numeric(df[numeric_candidate], errors='coerce')

threshold = df[numeric_candidate].mean() if not np.isnan(df[numeric_candidate].mean()) else 10
filtered_df = df[df[numeric_candidate] > threshold]
print(f"Filtered records with {numeric_candidate} > {threshold:.2f} (Mean value): {len(filtered_df)} records\n")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_candidate}_normalized"] = (
    filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()
) / (filtered_df[numeric_candidate].std() if filtered_df[numeric_candidate].std() else 1)

print(f"Normalized '{numeric_candidate}' for filtered records:\n")
display(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

# Try grouping by a categorical field if one exists (not the numeric field)
group_field = None
for col in df.columns:
    if (df[col].dtype == object or df[col].dtype.name == 'category') and col != numeric_candidate:
        nunique = df[col].nunique(dropna=True)
        if 2 < nunique < 40:
            group_field = col
            break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean().reset_index().sort_values(numeric_candidate, ascending=False)
    print(f"\nGrouped mean of '{numeric_candidate}' by '{group_field}':")
    display(grouped_df.head())
else:
    print('\nNo suitable categorical group field found in this record set for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_candidate].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_candidate}")
plt.xlabel(numeric_candidate)
plt.ylabel('Frequency')
plt.show()

# If grouping field exists, barplot of mean by group
if group_field:
    plt.figure(figsize=(10,4))
    order = grouped_df[group_field]
    sns.barplot(data=grouped_df, x=group_field, y=numeric_candidate, order=order)
    plt.title(f"Mean {numeric_candidate} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_candidate}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to load a Croissant-compliant dataset using `mlcroissant`, review record sets and fields by their `@id`, and perform basic exploratory data analysis (EDA) and visualization.
- Strong use of the record set and field `@id`s throughout ensures reproducible and standards-based data access.
- Further analyses, such as biomarker discovery or advanced statistical analysis, can be built with these foundations.

**Next steps:** Integrate these operations into FAIR-aligned pipelines, or export processed data for downstream computational biology or machine learning workflows.
